In [ ]:
# Exported from analysis.ipynb

In [ ]:
# Exported from analysis.ipynb

In [ ]:
# Exported from analysis.ipynb

# the edge-energy autocorrelation analysis: Figure 2 edge-energy autocorrelation localization
Compute line-graph autocorrelation and high-energy quantile localization
statistics from stability-DMS edge Dirichlet energies generated by the Figure 2 stability-DMS experiment /
the primary stability-DMS run.

In [ ]:
from pathlib import Path
import json
import os
import sys

import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import linregress, pearsonr, spearmanr, wilcoxon

os.environ.setdefault("MPLBACKEND", "Agg")
import matplotlib.pyplot as plt

analysis_dir = Path(__file__).resolve().parents[1]
scripts_dir = analysis_dir.parents[1] / "scripts"
sys.path.insert(0, str(scripts_dir))
from paper_runtime import find_project_root, _patch_matplotlib_savefig

project_root = find_project_root(analysis_dir)
processed_dir = project_root / "data" / "processed" / "stability_dms"
figure_dir = analysis_dir / "figures" / "figure_2"
table_dir = analysis_dir / "tables"
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

try:
    _patch_matplotlib_savefig()
except Exception:
    pass

plt.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.8,
    }
)

## Load edge-level Dirichlet energies
The upstream the Figure 2 stability-DMS experiment/the primary stability-DMS run table records one row per DMS graph edge with
the squared fitness difference (`edge_energy`) between adjacent variants. For
spatial autocorrelation, we work on the line graph: each original graph edge
becomes an observation, and two observations are neighbors when the original
edges share a sequence node.

In [ ]:
edge_detail_path = processed_dir / "spectral_partition_boundary_tmap_edge_details.csv"
if not edge_detail_path.exists():
    raise FileNotFoundError(edge_detail_path)

edge_df = pd.read_csv(edge_detail_path, usecols=["file", "dataset", "u", "v", "edge_energy"])
edge_df = edge_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["edge_energy", "u", "v"])
edge_df["u"] = edge_df["u"].astype(np.int64)
edge_df["v"] = edge_df["v"].astype(np.int64)
edge_df["edge_energy"] = edge_df["edge_energy"].astype(float)

print(
    f"Loaded {len(edge_df):,} edge-energy rows from "
    f"{edge_df['file'].nunique()} stability-DMS domains."
)

## Spatial autocorrelation helpers
Moran's I and Geary's C are computed with row-standardized line-graph
weights. Row standardization prevents high-degree sequence nodes from
dominating the statistic simply because they create many adjacent edge pairs.
The continuous statistics use `log1p(edge_energy)` to reduce outlier leverage
while preserving the ordering and magnitude of Dirichlet energies.

In [ ]:
QUANTILES = (0.75, 0.90, 0.95, 0.99)
PANEL_WIDTH_IN = 5.4
PANEL_HEIGHT_IN = 3.6


def _incidence_lists(u: np.ndarray, v: np.ndarray) -> dict[int, list[int]]:
    incidence: dict[int, list[int]] = {}
    for edge_idx, (node_u, node_v) in enumerate(zip(u, v)):
        incidence.setdefault(int(node_u), []).append(edge_idx)
        incidence.setdefault(int(node_v), []).append(edge_idx)
    return incidence


def _row_standardized_line_graph_moments(values: np.ndarray, incidence: dict[int, list[int]]) -> dict[str, float]:
    x = np.asarray(values, dtype=float)
    n_edges = int(x.size)
    centered = x - float(np.mean(x))
    denom = float(np.dot(centered, centered))

    line_degree = np.zeros(n_edges, dtype=np.int64)
    neighbor_centered_sum = np.zeros(n_edges, dtype=float)
    neighbor_sqdiff_sum = np.zeros(n_edges, dtype=float)
    undirected_line_pairs = 0

    for incident_edges in incidence.values():
        degree = len(incident_edges)
        if degree < 2:
            continue
        idx = np.asarray(incident_edges, dtype=np.int64)
        undirected_line_pairs += degree * (degree - 1) // 2
        line_degree[idx] += degree - 1

        x_i = x[idx]
        centered_i = centered[idx]
        sum_x = float(np.sum(x_i))
        sum_x2 = float(np.dot(x_i, x_i))
        sum_centered = float(np.sum(centered_i))

        neighbor_centered_sum[idx] += sum_centered - centered_i
        neighbor_sqdiff_sum[idx] += (
            (degree - 1) * x_i * x_i
            - 2.0 * x_i * (sum_x - x_i)
            + (sum_x2 - x_i * x_i)
        )

    has_neighbors = line_degree > 0
    if denom <= 0 or not np.any(has_neighbors):
        return {
            "n_edges": n_edges,
            "n_line_pairs": int(undirected_line_pairs),
            "moran_i": np.nan,
            "geary_c": np.nan,
            "line_degree_mean": float(np.mean(line_degree[has_neighbors])) if np.any(has_neighbors) else np.nan,
        }

    weighted_neighbor_mean = neighbor_centered_sum[has_neighbors] / line_degree[has_neighbors]
    moran_i = float(np.sum(centered[has_neighbors] * weighted_neighbor_mean) / denom)

    weighted_sqdiff_mean = neighbor_sqdiff_sum[has_neighbors] / line_degree[has_neighbors]
    geary_c = float(((n_edges - 1) / (2.0 * n_edges)) * np.sum(weighted_sqdiff_mean) / denom)

    return {
        "n_edges": n_edges,
        "n_line_pairs": int(undirected_line_pairs),
        "moran_i": moran_i,
        "geary_c": geary_c,
        "line_degree_mean": float(np.mean(line_degree[has_neighbors])),
    }


def _quantile_neighbor_stats(
    edge_energy: np.ndarray,
    incidence: dict[int, list[int]],
    quantile: float,
) -> dict[str, float]:
    energy = np.asarray(edge_energy, dtype=float)
    threshold = float(np.quantile(energy, quantile))
    high = energy >= threshold
    n_edges = int(energy.size)
    n_high = int(np.sum(high))

    line_degree = np.zeros(n_edges, dtype=np.int64)
    high_neighbor_count = np.zeros(n_edges, dtype=np.int64)

    for incident_edges in incidence.values():
        degree = len(incident_edges)
        if degree < 2:
            continue
        idx = np.asarray(incident_edges, dtype=np.int64)
        n_incident_high = int(np.sum(high[idx]))
        line_degree[idx] += degree - 1
        high_neighbor_count[idx] += n_incident_high - high[idx].astype(np.int64)

    high_with_neighbors = high & (line_degree > 0)
    if not np.any(high_with_neighbors) or n_edges <= 1 or n_high <= 1:
        observed_neighbor_high_probability = np.nan
        expected_neighbor_high_probability = np.nan
        neighbor_high_enrichment = np.nan
    else:
        observed_neighbor_high_probability = float(
            np.mean(high_neighbor_count[high_with_neighbors] / line_degree[high_with_neighbors])
        )
        expected_neighbor_high_probability = float((n_high - 1) / (n_edges - 1))
        neighbor_high_enrichment = observed_neighbor_high_probability / expected_neighbor_high_probability

    binary_stats = _row_standardized_line_graph_moments(high.astype(float), incidence)
    return {
        "energy_quantile": quantile,
        "energy_threshold": threshold,
        "n_high_edges": n_high,
        "high_edge_fraction": float(n_high / n_edges),
        "observed_neighbor_high_probability": observed_neighbor_high_probability,
        "expected_neighbor_high_probability": expected_neighbor_high_probability,
        "neighbor_high_enrichment": neighbor_high_enrichment,
        "binary_moran_i": binary_stats["moran_i"],
        "binary_geary_c": binary_stats["geary_c"],
    }


def _safe_wilcoxon(values: pd.Series, *, alternative: str) -> float:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    clean = clean.loc[clean != 0]
    if len(clean) == 0:
        return float("nan")
    return float(wilcoxon(clean, alternative=alternative).pvalue)


def _format_pvalue(pvalue: float) -> str:
    if not np.isfinite(pvalue):
        return "NA"
    if pvalue < 1e-3:
        return f"{pvalue:.1e}"
    return f"{pvalue:.3f}"


def _association_stats(table: pd.DataFrame, x_col: str, y_col: str) -> dict[str, float]:
    clean = table[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 3:
        return {
            "n": int(len(clean)),
            "pearson_r": float("nan"),
            "pearson_p": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_p": float("nan"),
            "linear_slope_per_t_map": float("nan"),
            "linear_intercept": float("nan"),
            "linear_p": float("nan"),
        }

    x = clean[x_col].to_numpy(dtype=float)
    y = clean[y_col].to_numpy(dtype=float)
    pearson = pearsonr(x, y)
    spearman = spearmanr(x, y)
    linear = linregress(x, y)
    return {
        "n": int(len(clean)),
        "pearson_r": float(pearson.statistic),
        "pearson_p": float(pearson.pvalue),
        "spearman_rho": float(spearman.statistic),
        "spearman_p": float(spearman.pvalue),
        "linear_slope_per_t_map": float(linear.slope),
        "linear_intercept": float(linear.intercept),
        "linear_p": float(linear.pvalue),
    }


def _plot_linear_fit_with_bootstrap_ci(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    *,
    color: str,
    seed: int,
    n_bootstrap: int = 1000,
) -> None:
    clean = np.isfinite(x) & np.isfinite(y)
    x = x[clean]
    y = y[clean]
    if len(x) < 3:
        return

    x_grid = np.linspace(float(np.min(x)), float(np.max(x)), 120)
    linear = linregress(x, y)
    y_fit = linear.intercept + linear.slope * x_grid

    rng = np.random.default_rng(seed)
    boot_predictions = np.empty((n_bootstrap, len(x_grid)), dtype=float)
    for boot_idx in range(n_bootstrap):
        sample = rng.integers(0, len(x), len(x))
        sample_fit = linregress(x[sample], y[sample])
        boot_predictions[boot_idx] = sample_fit.intercept + sample_fit.slope * x_grid

    lo, hi = np.nanpercentile(boot_predictions, [2.5, 97.5], axis=0)
    ax.fill_between(x_grid, lo, hi, color=color, alpha=0.14, linewidth=0, zorder=1)
    ax.plot(x_grid, y_fit, color=color, linewidth=1.45, zorder=3)

## Compute domain-level statistics

In [ ]:
domain_rows: list[dict[str, object]] = []
quantile_rows: list[dict[str, object]] = []

for (file_name, dataset), group in edge_df.groupby(["file", "dataset"], sort=False):
    u = group["u"].to_numpy(dtype=np.int64)
    v = group["v"].to_numpy(dtype=np.int64)
    edge_energy = group["edge_energy"].to_numpy(dtype=float)
    incidence = _incidence_lists(u, v)

    log_energy = np.log1p(edge_energy)
    continuous = _row_standardized_line_graph_moments(log_energy, incidence)
    continuous.update(
        {
            "file": file_name,
            "dataset": dataset,
            "energy_transform": "log1p(edge_energy)",
            "moran_i_expected_random": float(-1.0 / (len(edge_energy) - 1)) if len(edge_energy) > 1 else np.nan,
            "geary_c_expected_random": 1.0,
        }
    )
    continuous["moran_i_excess_over_random"] = (
        continuous["moran_i"] - continuous["moran_i_expected_random"]
    )
    continuous["one_minus_geary_c"] = 1.0 - continuous["geary_c"]
    domain_rows.append(continuous)

    for quantile in QUANTILES:
        qrow = _quantile_neighbor_stats(edge_energy, incidence, quantile)
        qrow.update({"file": file_name, "dataset": dataset})
        quantile_rows.append(qrow)

domain_summary = pd.DataFrame(domain_rows)
quantile_summary = pd.DataFrame(quantile_rows)

domain_summary_path = table_dir / "edge_energy_autocorrelation_domain_summary.csv"
quantile_summary_path = table_dir / "edge_energy_quantile_adjacency_summary.csv"
domain_summary.to_csv(domain_summary_path, index=False)
quantile_summary.to_csv(quantile_summary_path, index=False)

print(f"Domain autocorrelation table written to: {domain_summary_path}")
print(f"Quantile adjacency table written to: {quantile_summary_path}")

## Population summaries

In [ ]:
population = {
    "n_domains": int(len(domain_summary)),
    "n_edges_total": int(domain_summary["n_edges"].sum()),
    "panel_width_inches": PANEL_WIDTH_IN,
    "panel_height_inches": PANEL_HEIGHT_IN,
    "continuous_energy_transform": "log1p(edge_energy)",
    "line_graph_weighting": "row-standardized binary adjacency between graph edges sharing a sequence node",
    "moran_i_median": float(domain_summary["moran_i"].median()),
    "moran_i_min": float(domain_summary["moran_i"].min()),
    "moran_i_max": float(domain_summary["moran_i"].max()),
    "moran_i_excess_over_random_wilcoxon_p": _safe_wilcoxon(
        domain_summary["moran_i_excess_over_random"],
        alternative="greater",
    ),
    "geary_c_median": float(domain_summary["geary_c"].median()),
    "geary_c_min": float(domain_summary["geary_c"].min()),
    "geary_c_max": float(domain_summary["geary_c"].max()),
    "geary_c_less_than_one_wilcoxon_p": _safe_wilcoxon(
        domain_summary["geary_c"] - 1.0,
        alternative="less",
    ),
    "quantile_neighbor_enrichment": {},
}

for quantile, group in quantile_summary.groupby("energy_quantile"):
    enrichment = group["neighbor_high_enrichment"]
    population["quantile_neighbor_enrichment"][str(float(quantile))] = {
        "median": float(enrichment.median()),
        "min": float(enrichment.min()),
        "max": float(enrichment.max()),
        "wilcoxon_p_greater_than_one": _safe_wilcoxon(enrichment - 1.0, alternative="greater"),
        "median_observed_neighbor_high_probability": float(group["observed_neighbor_high_probability"].median()),
        "median_expected_neighbor_high_probability": float(group["expected_neighbor_high_probability"].median()),
    }

population_summary_path = table_dir / "edge_energy_autocorrelation_population_summary.json"
with population_summary_path.open("w") as handle:
    json.dump(population, handle, indent=2, sort_keys=True)

print(json.dumps(population, indent=2, sort_keys=True))

## tMAP-autocorrelation association

Link the continuous line-graph autocorrelation statistics to the independently
computed spectral-boundary `t_MAP` values. Lower `t_MAP` indicates a more
rugged landscape, so the expected relationship is negative when the
autocorrelation statistic is oriented so that larger values mean stronger
local energy clustering.

In [ ]:
tmap_summary_path = processed_dir / "spectral_partition_boundary_tmap_domain_summary.csv"
if not tmap_summary_path.exists():
    raise FileNotFoundError(tmap_summary_path)

tmap_summary = pd.read_csv(tmap_summary_path)
tmap_columns = [
    "file",
    "dataset",
    "observed_t_map",
    "n_nodes",
    "n_edges",
    "partition_balance",
    "cut_energy_enrichment",
]
tmap_ok = tmap_summary.loc[tmap_summary["status"].eq("ok"), tmap_columns].copy()
tmap_autocorr = domain_summary.merge(
    tmap_ok,
    on=["file", "dataset"],
    how="inner",
    suffixes=("_autocorr", "_tmap"),
    validate="one_to_one",
)
tmap_autocorr = (
    tmap_autocorr.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["observed_t_map", "moran_i", "one_minus_geary_c"])
    .sort_values("observed_t_map")
    .reset_index(drop=True)
)

tmap_autocorr_path = table_dir / "tmap_vs_edge_energy_autocorrelation.csv"
tmap_autocorr.to_csv(tmap_autocorr_path, index=False)

tmap_autocorr_stats = {
    "n_domains": int(len(tmap_autocorr)),
    "x_metric": "observed_t_map",
    "x_source": str(tmap_summary_path.relative_to(project_root)),
    "expected_direction": "negative; lower t_MAP should coincide with stronger local edge-energy autocorrelation",
    "moran_i": _association_stats(tmap_autocorr, "observed_t_map", "moran_i"),
    "one_minus_geary_c": _association_stats(tmap_autocorr, "observed_t_map", "one_minus_geary_c"),
}
tmap_autocorr_stats_path = table_dir / "tmap_vs_edge_energy_autocorrelation_stats.json"
with tmap_autocorr_stats_path.open("w") as handle:
    json.dump(tmap_autocorr_stats, handle, indent=2, sort_keys=True)

TMAP_AUTOCORR_WIDTH_MM = 94.0
TMAP_AUTOCORR_HEIGHT_MM = 42.0
TMAP_AUTOCORR_WIDTH_IN = TMAP_AUTOCORR_WIDTH_MM / 25.4
TMAP_AUTOCORR_HEIGHT_IN = TMAP_AUTOCORR_HEIGHT_MM / 25.4

fig, axes = plt.subplots(
    1,
    2,
    figsize=(TMAP_AUTOCORR_WIDTH_IN, TMAP_AUTOCORR_HEIGHT_IN),
    sharex=True,
    constrained_layout=True,
)
tmap_metrics = [
    ("moran_i", "Moran's I", "#4C78A8", 2026052801),
    ("one_minus_geary_c", "1 - Geary's C", "#E8A838", 2026052802),
]
x = tmap_autocorr["observed_t_map"].to_numpy(dtype=float)
for ax, (metric, ylabel, color, seed) in zip(axes, tmap_metrics):
    y = tmap_autocorr[metric].to_numpy(dtype=float)
    ax.scatter(
        x,
        y,
        s=15,
        color=color,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.25,
        zorder=4,
    )
    _plot_linear_fit_with_bootstrap_ci(ax, x, y, color=color, seed=seed)

    stats = tmap_autocorr_stats[metric]
    ax.text(
        0.05,
        0.08,
        (
            r"$\rho$="
            f"{stats['spearman_rho']:.2f}\n"
            r"$P$="
            f"{_format_pvalue(stats['spearman_p'])}"
        ),
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=6.2,
    )
    ax.set_xlabel(r"$t_{MAP}$")
    ax.set_ylabel(ylabel)
    ax.grid(axis="both", linestyle="--", linewidth=0.55, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_title("Ruggedness vs Moran", pad=3)
axes[1].set_title("Ruggedness vs Geary", pad=3)

tmap_autocorr_pdf = figure_dir / "tmap_vs_edge_energy_autocorrelation.pdf"
tmap_autocorr_png = figure_dir / "tmap_vs_edge_energy_autocorrelation.png"
fig.savefig(tmap_autocorr_pdf)
fig.savefig(tmap_autocorr_png, dpi=300)
plt.close(fig)

print(f"tMAP-autocorrelation figure written to: {tmap_autocorr_pdf}")
print(f"tMAP-autocorrelation table written to: {tmap_autocorr_path}")
print(json.dumps(tmap_autocorr_stats, indent=2, sort_keys=True))

## Node-label permutation null collection

Collect the Figure 2 stability-DMS experiment node-label permutation chunks and compare the observed autocorrelation statistics against nulls that preserve graph topology, degree structure, and the marginal fitness distribution.

In [ ]:
null_dir = processed_dir / "node_label_permutation_nulls"
autocorr_null_paths = sorted(null_dir.glob("chunk_*_node_label_autocorrelation_domain_null.csv"))
quantile_null_paths = sorted(null_dir.glob("chunk_*_node_label_autocorrelation_quantile_null.csv"))

if autocorr_null_paths:
    autocorr_null_df = pd.concat(
        [pd.read_csv(path) for path in autocorr_null_paths],
        ignore_index=True,
    )
    quantile_null_df = pd.concat(
        [pd.read_csv(path) for path in quantile_null_paths],
        ignore_index=True,
    ) if quantile_null_paths else pd.DataFrame()

    autocorr_null_path = table_dir / "edge_energy_autocorrelation_node_label_null_domain.csv"
    quantile_null_path = table_dir / "edge_energy_autocorrelation_node_label_null_quantile.csv"
    autocorr_null_df.to_csv(autocorr_null_path, index=False)
    if len(quantile_null_df) > 0:
        quantile_null_df.to_csv(quantile_null_path, index=False)

    observed = domain_summary.copy()
    observed = observed.loc[observed["status"].fillna("ok").eq("ok") if "status" in observed.columns else np.ones(len(observed), dtype=bool)]
    ok_null = autocorr_null_df.loc[autocorr_null_df["status"].eq("ok")].copy()

    empirical_rows = []
    for row in observed.itertuples(index=False):
        sub = ok_null.loc[ok_null["file"].eq(row.file)].copy()
        n_null = int(len(sub))
        if n_null == 0:
            continue
        empirical_rows.append(
            {
                "dataset": row.dataset,
                "file": row.file,
                "n_null": n_null,
                "observed_moran_i": float(row.moran_i),
                "null_moran_i_median": float(sub["moran_i"].median()),
                "empirical_p_moran_i_ge_observed": float((np.sum(sub["moran_i"] >= row.moran_i) + 1) / (n_null + 1)),
                "observed_geary_c": float(row.geary_c),
                "null_geary_c_median": float(sub["geary_c"].median()),
                "empirical_p_geary_c_le_observed": float((np.sum(sub["geary_c"] <= row.geary_c) + 1) / (n_null + 1)),
                "observed_one_minus_geary_c": float(row.one_minus_geary_c),
                "null_one_minus_geary_c_median": float(sub["one_minus_geary_c"].median()),
                "empirical_p_one_minus_geary_c_ge_observed": float((np.sum(sub["one_minus_geary_c"] >= row.one_minus_geary_c) + 1) / (n_null + 1)),
            }
        )

    autocorr_empirical_df = pd.DataFrame(empirical_rows)
    autocorr_empirical_path = table_dir / "edge_energy_autocorrelation_node_label_null_empirical_p.csv"
    autocorr_empirical_df.to_csv(autocorr_empirical_path, index=False)

    null_by_perm = ok_null.groupby("perm_idx", as_index=False).agg(
        median_moran_i=("moran_i", "median"),
        median_geary_c=("geary_c", "median"),
        median_one_minus_geary_c=("one_minus_geary_c", "median"),
    )
    observed_median_moran = float(observed["moran_i"].median())
    observed_median_geary = float(observed["geary_c"].median())
    observed_median_one_minus_geary = float(observed["one_minus_geary_c"].median())

    quantile_population = {}
    if len(quantile_null_df) > 0:
        observed_quantile = quantile_summary.groupby("energy_quantile")["neighbor_high_enrichment"].median()
        ok_quantile_null = quantile_null_df.loc[quantile_null_df["status"].eq("ok")].copy()
        null_quantile_by_perm = ok_quantile_null.groupby(["perm_idx", "energy_quantile"], as_index=False)["neighbor_high_enrichment"].median()
        for quantile, observed_value in observed_quantile.items():
            sub = null_quantile_by_perm.loc[null_quantile_by_perm["energy_quantile"].eq(quantile), "neighbor_high_enrichment"]
            if len(sub) == 0:
                continue
            quantile_population[str(float(quantile))] = {
                "observed_median_neighbor_high_enrichment": float(observed_value),
                "null_median_neighbor_high_enrichment": float(sub.median()),
                "empirical_p_null_ge_observed": float((np.sum(sub >= observed_value) + 1) / (len(sub) + 1)),
                "n_null_permutations": int(len(sub)),
            }

    autocorr_null_summary = {
        "null_type": "node-label permutation within each domain",
        "n_null_rows": int(len(ok_null)),
        "n_null_permutations": int(ok_null["perm_idx"].nunique()),
        "n_domains": int(observed["file"].nunique()),
        "observed_median_moran_i": observed_median_moran,
        "null_median_of_permutation_median_moran_i": float(null_by_perm["median_moran_i"].median()),
        "empirical_p_median_moran_i_ge_observed": float((np.sum(null_by_perm["median_moran_i"] >= observed_median_moran) + 1) / (len(null_by_perm) + 1)),
        "observed_median_geary_c": observed_median_geary,
        "null_median_of_permutation_median_geary_c": float(null_by_perm["median_geary_c"].median()),
        "empirical_p_median_geary_c_le_observed": float((np.sum(null_by_perm["median_geary_c"] <= observed_median_geary) + 1) / (len(null_by_perm) + 1)),
        "observed_median_one_minus_geary_c": observed_median_one_minus_geary,
        "null_median_of_permutation_median_one_minus_geary_c": float(null_by_perm["median_one_minus_geary_c"].median()),
        "empirical_p_median_one_minus_geary_c_ge_observed": float((np.sum(null_by_perm["median_one_minus_geary_c"] >= observed_median_one_minus_geary) + 1) / (len(null_by_perm) + 1)),
        "quantile_neighbor_enrichment": quantile_population,
        "source_files": [str(path) for path in autocorr_null_paths],
    }
    autocorr_null_summary_path = table_dir / "edge_energy_autocorrelation_node_label_null_summary.json"
    with autocorr_null_summary_path.open("w") as handle:
        json.dump(autocorr_null_summary, handle, indent=2, sort_keys=True)

    print(f"Collected {len(autocorr_null_paths)} autocorrelation null chunks from {null_dir}")
else:
    autocorr_null_df = pd.DataFrame()
    quantile_null_df = pd.DataFrame()
    print(f"No node-label autocorrelation null chunks found in {null_dir}; skipping null collection.")

## Main-text panel
The panel is sized to match the compact Figure 2 analytical panels.

In [ ]:
plot_domain = domain_summary.sort_values(["moran_i", "dataset"]).reset_index(drop=True)
plot_quantile = quantile_summary.copy()
plot_quantile["quantile_label"] = plot_quantile["energy_quantile"].map(
    {
        0.75: "Top 25%",
        0.90: "Top 10%",
        0.95: "Top 5%",
        0.99: "Top 1%",
    }
)
quantile_order = [0.75, 0.90, 0.95, 0.99]
x_quantile = {q: idx for idx, q in enumerate(quantile_order)}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(PANEL_WIDTH_IN, PANEL_HEIGHT_IN),
    gridspec_kw={"width_ratios": [0.95, 1.55], "wspace": 0.42},
    constrained_layout=True,
)
ax0, ax1 = axes

rng = np.random.default_rng(20260526)
autocorr_points = pd.DataFrame(
    {
        "metric": ["Moran's I"] * len(plot_domain) + ["1 - Geary's C"] * len(plot_domain),
        "value": list(plot_domain["moran_i"]) + list(plot_domain["one_minus_geary_c"]),
    }
)
metric_x = {"Moran's I": 0.0, "1 - Geary's C": 1.0}
metric_color = {"Moran's I": "#4C78A8", "1 - Geary's C": "#E8A838"}

for metric, group in autocorr_points.groupby("metric", sort=False):
    x = metric_x[metric] + rng.normal(0.0, 0.045, len(group))
    ax0.scatter(
        x,
        group["value"],
        s=20,
        color=metric_color[metric],
        alpha=0.78,
        edgecolor="white",
        linewidth=0.25,
        zorder=3,
    )
    median = float(group["value"].median())
    q25, q75 = np.quantile(group["value"], [0.25, 0.75])
    ax0.plot([metric_x[metric] - 0.18, metric_x[metric] + 0.18], [median, median], color="0.1", lw=1.4, zorder=4)
    ax0.plot([metric_x[metric], metric_x[metric]], [q25, q75], color="0.1", lw=1.1, zorder=4)

ax0.axhline(0.0, color="0.35", linestyle="--", linewidth=0.9, zorder=1)
ax0.set_xlim(-0.42, 1.42)
ax0.set_xticks([0, 1])
ax0.set_xticklabels(["Moran's I", "1 -\nGeary's C"])
ax0.set_ylabel("Line-graph autocorrelation")
ax0.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.28)
ax0.set_title("Autocorrelation", pad=4)

for _, group in plot_quantile.groupby("dataset", sort=False):
    xs = [x_quantile[float(q)] for q in group["energy_quantile"]]
    ys = group["neighbor_high_enrichment"].to_numpy(dtype=float)
    ax1.plot(xs, ys, color="0.72", linewidth=0.75, alpha=0.55, zorder=1)

median_by_quantile = (
    plot_quantile.groupby("energy_quantile", sort=True)["neighbor_high_enrichment"]
    .median()
    .reindex(quantile_order)
)
q25_by_quantile = (
    plot_quantile.groupby("energy_quantile", sort=True)["neighbor_high_enrichment"]
    .quantile(0.25)
    .reindex(quantile_order)
)
q75_by_quantile = (
    plot_quantile.groupby("energy_quantile", sort=True)["neighbor_high_enrichment"]
    .quantile(0.75)
    .reindex(quantile_order)
)
xs = np.arange(len(quantile_order))
ax1.fill_between(xs, q25_by_quantile, q75_by_quantile, color="#4C78A8", alpha=0.18, linewidth=0, zorder=2)
ax1.plot(xs, median_by_quantile, color="#4C78A8", marker="o", markersize=4.5, linewidth=1.8, zorder=4)
ax1.axhline(1.0, color="0.35", linestyle="--", linewidth=0.9, zorder=1)
ax1.set_yscale("log", base=2)
ax1.set_ylim(0.8, max(32.0, float(plot_quantile["neighbor_high_enrichment"].max()) * 1.1))
ax1.set_yticks([1, 2, 4, 8, 16, 32])
ax1.set_yticklabels(["1", "2", "4", "8", "16", "32"])
ax1.set_xticks(xs)
ax1.set_xticklabels(["Top\n25%", "Top\n10%", "Top\n5%", "Top\n1%"])
ax1.set_xlabel("Edge-energy quantile")
ax1.set_ylabel("High-energy neighbor enrichment")
ax1.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.28)
ax1.set_title("High-energy adjacency", pad=4)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

panel_pdf = figure_dir / "edge_energy_autocorrelation_localization_panel.pdf"
panel_png = figure_dir / "edge_energy_autocorrelation_localization_panel.png"
fig.savefig(panel_pdf, bbox_inches="tight")
fig.savefig(panel_png, dpi=300, bbox_inches="tight")
plt.close(fig)

panel_spec = {
    "panel_width_inches": PANEL_WIDTH_IN,
    "panel_height_inches": PANEL_HEIGHT_IN,
    "pdf": str(panel_pdf.relative_to(analysis_dir)),
    "png": str(panel_png.relative_to(analysis_dir)),
}
panel_spec_path = table_dir / "edge_energy_autocorrelation_panel_spec.json"
with panel_spec_path.open("w") as handle:
    json.dump(panel_spec, handle, indent=2, sort_keys=True)

print(f"Main-text panel written to: {panel_pdf}")
print(f"Panel size: {PANEL_WIDTH_IN} x {PANEL_HEIGHT_IN} inches")

## Strongest-autocorrelation graph example
The strongest continuous autocorrelation landscape is shown as two separate
graph layers with the same topology-only layout: edge energy and spectral-cut
node classification.

In [ ]:
def _read_edge_details_for_file(file_name: str) -> pd.DataFrame:
    usecols = ["file", "dataset", "u", "v", "edge_energy", "is_cut", "same_side"]
    chunks = []
    for chunk in pd.read_csv(edge_detail_path, usecols=usecols, chunksize=500_000):
        sub = chunk.loc[chunk["file"].astype(str).eq(str(file_name))].copy()
        if len(sub) > 0:
            chunks.append(sub)
    if not chunks:
        raise RuntimeError(f"No edge details found for {file_name}.")
    out = pd.concat(chunks, ignore_index=True)
    out["u"] = out["u"].astype(np.int64)
    out["v"] = out["v"].astype(np.int64)
    out["edge_energy"] = out["edge_energy"].astype(float)
    out["is_cut"] = out["is_cut"].astype(bool)
    out["same_side"] = out["same_side"].astype(bool)
    return out


def _infer_cut_side_labels(edge_table: pd.DataFrame, n_nodes: int) -> tuple[np.ndarray, int]:
    constraints: list[list[tuple[int, int]]] = [[] for _ in range(n_nodes)]
    for row in edge_table.itertuples(index=False):
        parity = 0 if bool(row.same_side) else 1
        u = int(row.u)
        v = int(row.v)
        constraints[u].append((v, parity))
        constraints[v].append((u, parity))

    labels = np.full(n_nodes, -1, dtype=np.int8)
    conflicts = 0
    for start in range(n_nodes):
        if labels[start] >= 0:
            continue
        labels[start] = 0
        stack = [start]
        while stack:
            current = stack.pop()
            for neighbor, parity in constraints[current]:
                expected = int(labels[current]) ^ int(parity)
                if labels[neighbor] < 0:
                    labels[neighbor] = expected
                    stack.append(neighbor)
                elif int(labels[neighbor]) != expected:
                    conflicts += 1
    if conflicts:
        raise RuntimeError(f"Spectral cut side constraints contain {conflicts} conflicts.")
    return labels, conflicts


def _get_graph_layout(graph: nx.Graph, cache_key: tuple[str, str], seed: int = 20260528) -> dict[int, np.ndarray]:
    cache = globals().setdefault("_autocorr_graph_layout_cache", {})
    if cache_key in cache:
        return cache[cache_key]

    pos = None
    layout_source = "networkx spectral_layout initialized spring_layout"
    for module_name in ("networkx.drawing.nx_agraph", "networkx.drawing.nx_pydot"):
        try:
            graphviz_layout = __import__(module_name, fromlist=["graphviz_layout"]).graphviz_layout
            pos = graphviz_layout(graph, prog="sfdp")
            layout_source = f"{module_name}.graphviz_layout(prog='sfdp')"
            break
        except Exception:
            continue

    if pos is None:
        try:
            pos = nx.spectral_layout(graph, dim=2)
            pos = nx.spring_layout(graph, seed=seed, pos=pos, iterations=45)
        except Exception:
            pos = nx.spring_layout(graph, seed=seed, iterations=90)

    pos = {int(node): np.asarray(coords, dtype=float)[:2] for node, coords in pos.items()}
    cache[cache_key] = (pos, layout_source)
    return cache[cache_key]


strongest_row = domain_summary.sort_values(["moran_i", "one_minus_geary_c"], ascending=False).iloc[0]
strongest_file = str(strongest_row["file"])
strongest_dataset = str(strongest_row["dataset"])
strongest_edges = _read_edge_details_for_file(strongest_file)
strongest_u = strongest_edges["u"].to_numpy(dtype=np.int64)
strongest_v = strongest_edges["v"].to_numpy(dtype=np.int64)
strongest_energy = strongest_edges["edge_energy"].to_numpy(dtype=float)
strongest_cut_mask = strongest_edges["is_cut"].to_numpy(dtype=bool)
strongest_n_nodes = int(max(strongest_u.max(), strongest_v.max()) + 1)
strongest_n_edges = int(len(strongest_edges))

strongest_graph = nx.Graph()
strongest_graph.add_nodes_from(range(strongest_n_nodes))
strongest_graph.add_edges_from(zip(strongest_u.tolist(), strongest_v.tolist()))
cut_side_labels, cut_side_conflicts = _infer_cut_side_labels(strongest_edges, strongest_n_nodes)
layout_pos, layout_source = _get_graph_layout(
    strongest_graph,
    cache_key=(strongest_file, "strongest_autocorrelation_graph"),
)

pos_array = np.vstack([layout_pos[node] for node in range(strongest_n_nodes)])
edge_segments = np.stack([pos_array[strongest_u], pos_array[strongest_v]], axis=1)
edge_order = np.argsort(strongest_energy)
energy_log10 = np.log10(np.maximum(strongest_energy, 1e-12))
energy_norm = plt.Normalize(
    vmin=float(np.nanpercentile(energy_log10, 5)),
    vmax=float(np.nanpercentile(energy_log10, 99)),
)
side_palette = {0: "#4C78A8", 1: "#E45756"}
node_size = float(np.clip(260.0 / np.sqrt(max(strongest_n_nodes, 1)), 3.0, 9.0))
EXAMPLE_GRAPH_WIDTH_MM = 47.0
EXAMPLE_GRAPH_HEIGHT_MM = 42.0
EXAMPLE_GRAPH_WIDTH_IN = EXAMPLE_GRAPH_WIDTH_MM / 25.4
EXAMPLE_GRAPH_HEIGHT_IN = EXAMPLE_GRAPH_HEIGHT_MM / 25.4

xmin, ymin = np.nanmin(pos_array, axis=0)
xmax, ymax = np.nanmax(pos_array, axis=0)
xpad = 0.05 * (xmax - xmin if xmax > xmin else 1.0)
ypad = 0.05 * (ymax - ymin if ymax > ymin else 1.0)

fig = plt.figure(figsize=(EXAMPLE_GRAPH_WIDTH_IN, EXAMPLE_GRAPH_HEIGHT_IN))
ax = fig.add_axes([0.01, 0.03, 0.74, 0.94])
energy_collection = LineCollection(
    edge_segments[edge_order],
    cmap=plt.cm.plasma,
    norm=energy_norm,
    linewidths=0.28,
    alpha=0.58,
    rasterized=True,
    zorder=1,
)
energy_collection.set_array(energy_log10[edge_order])
ax.add_collection(energy_collection)
ax.scatter(
    pos_array[:, 0],
    pos_array[:, 1],
    s=node_size * 0.45,
    color="0.70",
    alpha=0.55,
    linewidth=0,
    rasterized=True,
    zorder=2,
)
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)
ax.set_aspect("equal")
ax.set_axis_off()
cax = fig.add_axes([0.80, 0.18, 0.05, 0.66])
cbar = fig.colorbar(energy_collection, cax=cax)
cbar.ax.set_title(r"$\log_{10}E$", fontsize=5.5, pad=2)
cbar.ax.tick_params(labelsize=5.5, width=0.6, length=2)
energy_graph_pdf = figure_dir / "strongest_autocorrelation_example_edge_energy_plasma.pdf"
energy_graph_png = figure_dir / "strongest_autocorrelation_example_edge_energy_plasma.png"
fig.savefig(energy_graph_pdf)
fig.savefig(energy_graph_png, dpi=300)
plt.close(fig)

fig = plt.figure(figsize=(EXAMPLE_GRAPH_WIDTH_IN, EXAMPLE_GRAPH_HEIGHT_IN))
ax = fig.add_axes([0.01, 0.16, 0.98, 0.83])
noncut_collection = LineCollection(
    edge_segments[~strongest_cut_mask],
    colors="0.78",
    linewidths=0.15,
    alpha=0.18,
    rasterized=True,
    zorder=1,
)
ax.add_collection(noncut_collection)
cut_collection = LineCollection(
    edge_segments[strongest_cut_mask],
    colors="0.08",
    linewidths=0.70,
    alpha=0.58,
    rasterized=True,
    zorder=2,
)
ax.add_collection(cut_collection)
node_colors = [side_palette[int(label)] for label in cut_side_labels]
ax.scatter(
    pos_array[:, 0],
    pos_array[:, 1],
    s=node_size,
    c=node_colors,
    alpha=0.92,
    linewidth=0,
    rasterized=True,
    zorder=3,
)
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)
ax.set_aspect("equal")
ax.set_axis_off()
legend_handles = [
    Patch(facecolor=side_palette[0], edgecolor="none", label="Side 0"),
    Patch(facecolor=side_palette[1], edgecolor="none", label="Side 1"),
    Line2D([0], [0], color="0.08", linewidth=1.2, label="Cut"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.015),
    ncol=3,
    frameon=False,
    fontsize=5.5,
    handlelength=1.1,
    columnspacing=0.45,
)
classification_graph_pdf = figure_dir / "strongest_autocorrelation_example_cut_classification.pdf"
classification_graph_png = figure_dir / "strongest_autocorrelation_example_cut_classification.png"
fig.savefig(classification_graph_pdf)
fig.savefig(classification_graph_png, dpi=300)
plt.close(fig)

example_graph_spec = {
    "dataset": strongest_dataset,
    "file": strongest_file,
    "selection_rule": "largest continuous edge-energy autocorrelation by Moran's I in the edge-energy autocorrelation analysis",
    "moran_i": float(strongest_row["moran_i"]),
    "geary_c": float(strongest_row["geary_c"]),
    "one_minus_geary_c": float(strongest_row["one_minus_geary_c"]),
    "n_nodes": strongest_n_nodes,
    "n_edges": strongest_n_edges,
    "n_cut_edges": int(strongest_cut_mask.sum()),
    "panel_width_mm": EXAMPLE_GRAPH_WIDTH_MM,
    "panel_height_mm": EXAMPLE_GRAPH_HEIGHT_MM,
    "cut_side_counts": {
        "0": int(np.sum(cut_side_labels == 0)),
        "1": int(np.sum(cut_side_labels == 1)),
    },
    "cut_side_constraint_conflicts": int(cut_side_conflicts),
    "layout": layout_source,
    "energy_transform": "log10(edge_energy)",
    "energy_colormap": "plasma",
    "energy_pdf": str(energy_graph_pdf.relative_to(analysis_dir)),
    "energy_png": str(energy_graph_png.relative_to(analysis_dir)),
    "classification_pdf": str(classification_graph_pdf.relative_to(analysis_dir)),
    "classification_png": str(classification_graph_png.relative_to(analysis_dir)),
}
example_graph_spec_path = table_dir / "strongest_autocorrelation_example_graph_spec.json"
with example_graph_spec_path.open("w") as handle:
    json.dump(example_graph_spec, handle, indent=2, sort_keys=True)

print(f"Strongest-autocorrelation edge-energy graph written to: {energy_graph_pdf}")
print(f"Strongest-autocorrelation cut-classification graph written to: {classification_graph_pdf}")